In [0]:
import dlt
from pyspark.sql.functions import col, current_timestamp

In [0]:
bronze_schema_name = spark.conf.get("pipeline.bronze_schema_name", "weather_bronze")
silver_schema_name = spark.conf.get("pipeline.silver_schema_name", "weather_silver")

In [0]:
# Data Quality Rules
rules = {
    "valid_timestamp": "event_time IS NOT NULL",    
    "valid_city": "city IN ('Kraków', 'Warszawa', 'Wrocław', 'Poznań', 'Gdańsk', 'Łódź', 'Szczecin')",
    "valid_temperature": "temperature BETWEEN -100 AND 100",
    "non_negative_wind": "wind_speed >= 0"
}

# Dynamically building filtering conditions
valid_condition = " AND ".join(rules.values())
invalid_condition = f"NOT ({valid_condition})"

# Downloading and projecting raw data
@dlt.view(name="weather_raw_view")
def weather_raw_view():
    return (
        dlt.read_stream(f"{bronze_schema_name}.weather_streaming_data")
        .select(
            col("city"),
            col("lat"),
            col("lon"),
            col("event_timestamp").cast("timestamp").alias("event_time"),
            col("temperature"),
            col("wind_direction"),
            col("wind_speed"),
            col("air_quality"),
            col("pm10"),
            col("pm2_5"),
            col("source_filename"),
            col("ingestion_timestamp").alias("bronze_ingestion_time"),
            current_timestamp().alias("silver_transformation_time")
        )
    )

@dlt.view(name="weather_with_dq")
@dlt.expect("valid_timestamp", "event_time IS NOT NULL")
@dlt.expect("valid_city", "city IN ('Kraków', 'Warszawa', 'Wrocław', 'Poznań', 'Gdańsk', 'Łódź', 'Szczecin')")
@dlt.expect("valid_temperature", "temperature BETWEEN -100 AND 100")
@dlt.expect("non_negative_wind", "wind_speed >= 0")
def weather_with_dq():
    return dlt.read_stream("weather_raw_view")

# Quarantine table
@dlt.table(
    name=f"{bronze_schema_name}.weather_quarantine",
    comment="Quarantine table for records failing data quality rules."
)
def weather_quarantine():
    return dlt.read_stream("weather_with_dq").filter(invalid_condition)

# SILVER: A view that stores only valid records
@dlt.view(name="weather_valid")
def weather_valid():
    return dlt.read_stream("weather_with_dq").filter(valid_condition)

# Deduplication
dlt.create_streaming_table(
    f"{silver_schema_name}.unified_weather",  
    table_properties={"quality": "silver"}
)

dlt.apply_changes(
    target=f"{silver_schema_name}.unified_weather",
    source="weather_valid",
    keys=["city", "event_time"],
    sequence_by="bronze_ingestion_time",
    apply_as_deletes=None,
    except_column_list=[]
)